In [59]:
# Install mrjob if not already installed
!pip install mrjob -q

In [60]:
from collections import defaultdict
import itertools

---
## Q1. Word Count
Count the frequency of each word.

**Input:**
```
hadoop is fast
hadoop is scalable
```

In [61]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q1 = ["hadoop is fast", "hadoop is scalable"]

# MAPPER: emit (word, 1) for every word
def mapper_word_count(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append((word.lower(), 1))
    return pairs

# SHUFFLE: group values by key
def shuffle(pairs):
    grouped = defaultdict(list)
    for key, value in pairs:
        grouped[key].append(value)
    return dict(grouped)

# REDUCER: sum counts
def reducer_word_count(grouped):
    return {key: sum(values) for key, values in grouped.items()}

mapped   = mapper_word_count(input_q1)
shuffled = shuffle(mapped)
result   = reducer_word_count(shuffled)

print("Q1 (A) Word Count (Manual):")
for word, count in sorted(result.items()):
    print(f"  {word}: {count}")

Q1 (A) Word Count (Manual):
  fast: 1
  hadoop: 2
  is: 2
  scalable: 1


In [62]:
%%writefile q1_word_count.py
from mrjob.job import MRJob

class MRWordCount(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == '__main__':
    MRWordCount.run()

Overwriting q1_word_count.py


In [63]:
# Write input to a file and run mrjob
with open('input_q1.txt', 'w') as f:
    f.write("hadoop is fast\nhadoop is scalable\n")

print("Q1 (B) Word Count (mrjob):")
!python q1_word_count.py input_q1.txt

Q1 (B) Word Count (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q1_word_count.root.20260425.095339.340428
Running step 1 of 1...
job output is in /tmp/q1_word_count.root.20260425.095339.340428/output
Streaming final output from /tmp/q1_word_count.root.20260425.095339.340428/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/q1_word_count.root.20260425.095339.340428...


---
## Q2. Character Count
Count the frequency of each character (ignore spaces).

**Input:** `big data`

In [64]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q2 = ["big data"]

def mapper_char_count(lines):
    pairs = []
    for line in lines:
        for ch in line:
            if ch != ' ':
                pairs.append((ch.lower(), 1))
    return pairs

def reducer_char_count(grouped):
    return {key: sum(values) for key, values in grouped.items()}

mapped   = mapper_char_count(input_q2)
shuffled = shuffle(mapped)
result   = reducer_char_count(shuffled)

print("Q2 (A) Character Count (Manual):")
for ch, count in sorted(result.items()):
    print(f"  '{ch}': {count}")

Q2 (A) Character Count (Manual):
  'a': 2
  'b': 1
  'd': 1
  'g': 1
  'i': 1
  't': 1


In [65]:
%%writefile q2_char_count.py
from mrjob.job import MRJob

class MRCharCount(MRJob):

    def mapper(self, _, line):
        for ch in line.strip():
            if ch != ' ':
                yield ch.lower(), 1

    def reducer(self, ch, counts):
        yield ch, sum(counts)

if __name__ == '__main__':
    MRCharCount.run()

Overwriting q2_char_count.py


In [66]:
with open('input_q2.txt', 'w') as f:
    f.write("big data\n")

print("Q2 (B) Character Count (mrjob):")
!python q2_char_count.py input_q2.txt

Q2 (B) Character Count (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2_char_count.root.20260425.095339.664742
Running step 1 of 1...
job output is in /tmp/q2_char_count.root.20260425.095339.664742/output
Streaming final output from /tmp/q2_char_count.root.20260425.095339.664742/output...
"b"	1
"d"	1
"g"	1
"i"	1
"t"	1
"a"	2
Removing temp directory /tmp/q2_char_count.root.20260425.095339.664742...


---
## Q3. Average Word Length (Per Word)
Compute the average length of each unique word.

**Input:** `data science data big data`

In [67]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q3 = ["data science data big data"]

def mapper_avg_word_len(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append((word.lower(), len(word)))
    return pairs

def reducer_avg_word_len(grouped):
    return {key: sum(values) / len(values) for key, values in grouped.items()}

mapped   = mapper_avg_word_len(input_q3)
shuffled = shuffle(mapped)
result   = reducer_avg_word_len(shuffled)

print("Q3 (A) Average Word Length Per Word (Manual):")
for word, avg in sorted(result.items()):
    print(f"  {word}: {avg:.2f}")

Q3 (A) Average Word Length Per Word (Manual):
  big: 3.00
  data: 4.00
  science: 7.00


In [68]:
%%writefile q3_avg_word_len.py
from mrjob.job import MRJob

class MRAvgWordLen(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), len(word)

    def reducer(self, word, lengths):
        lengths = list(lengths)
        yield word, sum(lengths) / len(lengths)

if __name__ == '__main__':
    MRAvgWordLen.run()

Overwriting q3_avg_word_len.py


In [69]:
with open('input_q3.txt', 'w') as f:
    f.write("data science data big data\n")

print("Q3 (B) Average Word Length Per Word (mrjob):")
!python q3_avg_word_len.py input_q3.txt

Q3 (B) Average Word Length Per Word (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3_avg_word_len.root.20260425.095339.998070
Running step 1 of 1...
job output is in /tmp/q3_avg_word_len.root.20260425.095339.998070/output
Streaming final output from /tmp/q3_avg_word_len.root.20260425.095339.998070/output...
"science"	7.0
"big"	3.0
"data"	4.0
Removing temp directory /tmp/q3_avg_word_len.root.20260425.095339.998070...


---
## Q4. Global Average Word Length
Compute the average length of **all** words combined.

**Input:** `hadoop mapreduce spark`

In [70]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q4 = ["hadoop mapreduce spark"]

def mapper_global_avg(lines):
    pairs = []
    for line in lines:
        for word in line.strip().split():
            pairs.append(("global", len(word)))
    return pairs

def reducer_global_avg(grouped):
    return {key: sum(values) / len(values) for key, values in grouped.items()}

mapped   = mapper_global_avg(input_q4)
shuffled = shuffle(mapped)
result   = reducer_global_avg(shuffled)

print("Q4 (A) Global Average Word Length (Manual):")
print(f"  Average word length: {result['global']:.2f}")

Q4 (A) Global Average Word Length (Manual):
  Average word length: 6.67


In [71]:
%%writefile q4_global_avg.py
from mrjob.job import MRJob

class MRGlobalAvgWordLen(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            yield 'global', (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0
        for length, count in values:
            total_len   += length
            total_count += count
        yield key, total_len / total_count

if __name__ == '__main__':
    MRGlobalAvgWordLen.run()

Overwriting q4_global_avg.py


In [72]:
with open('input_q4.txt', 'w') as f:
    f.write("hadoop mapreduce spark\n")

print("Q4 (B) Global Average Word Length (mrjob):")
!python q4_global_avg.py input_q4.txt

Q4 (B) Global Average Word Length (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4_global_avg.root.20260425.095340.346882
Running step 1 of 1...
job output is in /tmp/q4_global_avg.root.20260425.095340.346882/output
Streaming final output from /tmp/q4_global_avg.root.20260425.095340.346882/output...
"global"	6.666666666666667
Removing temp directory /tmp/q4_global_avg.root.20260425.095340.346882...


---
## Q5. Q1–Q4 on External File + Top 5 Most Frequent Words

> Download the file from the Google Drive link provided in the assignment and save it as `input_q5.txt` in the same directory as this notebook.

The cell below reads `input_q5.txt` and runs all four analyses plus the Top-5 most frequent words.

In [73]:
# ── (A) Manual MapReduce on large file ───────────────────────────────────────
import os

FILE_Q5 = 'input_q5.txt'

if not os.path.exists(FILE_Q5):
    print("[!] Please download the file from the Google Drive link and save it as 'input_q5.txt'")
else:
    with open(FILE_Q5, 'r', encoding='utf-8', errors='ignore') as f:
        lines_q5 = f.readlines()

    # --- Q1 equivalent: Word Count ---
    wc_pairs   = mapper_word_count(lines_q5)
    wc_shuffle = shuffle(wc_pairs)
    wc_result  = reducer_word_count(wc_shuffle)

    print("=== Q5 – Word Count (sample, first 10) ===")
    for w, c in list(sorted(wc_result.items()))[:10]:
        print(f"  {w}: {c}")

    # --- Q2 equivalent: Character Count ---
    cc_pairs   = mapper_char_count(lines_q5)
    cc_shuffle = shuffle(cc_pairs)
    cc_result  = reducer_char_count(cc_shuffle)

    print("\n=== Q5 – Character Count (sample, first 10) ===")
    for ch, c in sorted(cc_result.items())[:10]:
        print(f"  '{ch}': {c}")

    # --- Q3 equivalent: Average Word Length Per Word ---
    awl_pairs   = mapper_avg_word_len(lines_q5)
    awl_shuffle = shuffle(awl_pairs)
    awl_result  = reducer_avg_word_len(awl_shuffle)

    print("\n=== Q5 – Avg Word Length Per Word (sample, first 10) ===")
    for w, avg in list(sorted(awl_result.items()))[:10]:
        print(f"  {w}: {avg:.2f}")

    # --- Q4 equivalent: Global Average Word Length ---
    gal_pairs   = mapper_global_avg(lines_q5)
    gal_shuffle = shuffle(gal_pairs)
    gal_result  = reducer_global_avg(gal_shuffle)

    print("\n=== Q5 – Global Average Word Length ===")
    print(f"  {gal_result['global']:.2f}")

    # --- Top 5 Most Frequent Words ---
    top5 = sorted(wc_result.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\n=== Q5 – Top 5 Most Frequent Words ===")
    for rank, (word, count) in enumerate(top5, 1):
        print(f"  #{rank}  {word}: {count}")

=== Q5 – Word Count (sample, first 10) ===
  ": 241
  "'tis: 1
  "a: 4
  "air,": 1
  "alas,: 1
  "amen": 2
  "amen"?: 1
  "amen,": 1
  "and: 1
  "aroint: 1

=== Q5 – Character Count (sample, first 10) ===
  '
': 124787
  '!': 8840
  '"': 488
  '#': 3
  '$': 2
  '%': 1
  '&': 21
  ''': 31077
  '(': 643
  ')': 644

=== Q5 – Avg Word Length Per Word (sample, first 10) ===
  ": 1.00
  "'tis: 5.00
  "a: 2.00
  "air,": 6.00
  "alas,: 6.00
  "amen": 6.00
  "amen"?: 7.00
  "amen,": 7.00
  "and: 4.00
  "aroint: 7.00

=== Q5 – Global Average Word Length ===
  4.48

=== Q5 – Top 5 Most Frequent Words ===
  #1  the: 27729
  #2  and: 26099
  #3  i: 19540
  #4  to: 18762
  #5  of: 18126


In [74]:
%%writefile q5_top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRTop5Words(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper_count, reducer=self.reducer_count),
            MRStep(mapper=self.mapper_swap,  reducer=self.reducer_top5)
        ]

    # Step 1 – count
    def mapper_count(self, _, line):
        for word in line.strip().split():
            yield word.lower(), 1

    def reducer_count(self, word, counts):
        yield word, sum(counts)

    # Step 2 – sort descending by count and emit top 5
    def mapper_swap(self, word, count):
        yield None, (count, word)

    def reducer_top5(self, _, word_counts):
        top5 = sorted(word_counts, key=lambda x: x[0], reverse=True)[:5]
        for count, word in top5:
            yield word, count

if __name__ == '__main__':
    MRTop5Words.run()

Overwriting q5_top5.py


In [75]:
import os
if os.path.exists('input_q5.txt'):
    print("Q5 (B) Top 5 Most Frequent Words (mrjob):")
    !python q5_top5.py input_q5.txt
else:
    print("[!] Place 'input_q5.txt' in this directory first.")

Q5 (B) Top 5 Most Frequent Words (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q5_top5.root.20260425.095343.269935
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/q5_top5.root.20260425.095343.269935/output
Streaming final output from /tmp/q5_top5.root.20260425.095343.269935/output...
"the"	27729
"and"	26099
"i"	19540
"to"	18762
"of"	18126
Removing temp directory /tmp/q5_top5.root.20260425.095343.269935...


---
## Q6. Average Marks Per Student

**Input:**
```
A 80
B 70
A 90
B 60
A 100
```

In [76]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q6 = ["A 80", "B 70", "A 90", "B 60", "A 100"]

def mapper_avg_marks(lines):
    pairs = []
    for line in lines:
        parts = line.strip().split()
        student, marks = parts[0], int(parts[1])
        pairs.append((student, marks))
    return pairs

def reducer_avg_marks(grouped):
    return {key: sum(values) / len(values) for key, values in grouped.items()}

mapped   = mapper_avg_marks(input_q6)
shuffled = shuffle(mapped)
result   = reducer_avg_marks(shuffled)

print("Q6 (A) Average Marks Per Student (Manual):")
for student, avg in sorted(result.items()):
    print(f"  Student {student}: {avg:.2f}")

Q6 (A) Average Marks Per Student (Manual):
  Student A: 90.00
  Student B: 65.00


In [77]:
%%writefile q6_avg_marks.py
from mrjob.job import MRJob

class MRAvgMarks(MRJob):

    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            student, marks = parts[0], int(parts[1])
            yield student, marks

    def reducer(self, student, marks):
        marks = list(marks)
        yield student, sum(marks) / len(marks)

if __name__ == '__main__':
    MRAvgMarks.run()

Overwriting q6_avg_marks.py


In [78]:
with open('input_q6.txt', 'w') as f:
    f.write("A 80\nB 70\nA 90\nB 60\nA 100\n")

print("Q6 (B) Average Marks Per Student (mrjob):")
!python q6_avg_marks.py input_q6.txt

Q6 (B) Average Marks Per Student (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6_avg_marks.root.20260425.095350.924203
Running step 1 of 1...
job output is in /tmp/q6_avg_marks.root.20260425.095350.924203/output
Streaming final output from /tmp/q6_avg_marks.root.20260425.095350.924203/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/q6_avg_marks.root.20260425.095350.924203...


---
## Q7. Average Salary Per Department + Highest Paid Department

**Input:**
```
HR 50000
IT 70000
HR 60000
IT 80000
```

In [79]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q7 = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

def mapper_avg_salary(lines):
    pairs = []
    for line in lines:
        parts = line.strip().split()
        dept, salary = parts[0], int(parts[1])
        pairs.append((dept, salary))
    return pairs

def reducer_avg_salary(grouped):
    return {key: sum(values) / len(values) for key, values in grouped.items()}

mapped   = mapper_avg_salary(input_q7)
shuffled = shuffle(mapped)
result   = reducer_avg_salary(shuffled)

print("Q7 (A) Average Salary Per Department (Manual):")
for dept, avg in sorted(result.items()):
    print(f"  {dept}: {avg:.2f}")

highest_dept = max(result, key=result.get)
print(f"\n  Highest Paid Department: {highest_dept} (Avg Salary: {result[highest_dept]:.2f})")

Q7 (A) Average Salary Per Department (Manual):
  HR: 55000.00
  IT: 75000.00

  Highest Paid Department: IT (Avg Salary: 75000.00)


In [80]:
%%writefile q7_avg_salary.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRAvgSalary(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper_salary,  reducer=self.reducer_avg),
            MRStep(mapper=self.mapper_find_max, reducer=self.reducer_max)
        ]

    # Step 1 – compute average per department
    def mapper_salary(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], int(parts[1])

    def reducer_avg(self, dept, salaries):
        salaries = list(salaries)
        avg = sum(salaries) / len(salaries)
        yield dept, avg

    # Step 2 – find highest paid department
    def mapper_find_max(self, dept, avg):
        yield 'max', (avg, dept)

    def reducer_max(self, _, dept_avgs):
        best_avg, best_dept = max(dept_avgs, key=lambda x: x[0])
        yield best_dept, f"Avg Salary = {best_avg:.2f} (Highest Paid)"

if __name__ == '__main__':
    MRAvgSalary.run()

Overwriting q7_avg_salary.py


In [81]:
with open('input_q7.txt', 'w') as f:
    f.write("HR 50000\nIT 70000\nHR 60000\nIT 80000\n")

print("Q7 (B) Average Salary Per Department + Highest Paid (mrjob):")
!python q7_avg_salary.py input_q7.txt

Q7 (B) Average Salary Per Department + Highest Paid (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q7_avg_salary.root.20260425.095351.262880
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/q7_avg_salary.root.20260425.095351.262880/output
Streaming final output from /tmp/q7_avg_salary.root.20260425.095351.262880/output...
"IT"	"Avg Salary = 75000.00 (Highest Paid)"
Removing temp directory /tmp/q7_avg_salary.root.20260425.095351.262880...


---
## Q8. Average Temperature Per City

**Input:**
```
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35
```

In [82]:
# ── (A) Manual MapReduce ──────────────────────────────────────────────────────

input_q8 = [
    "New York,38",
    "London,29",
    "Tokyo,35",
    "New York,32",
    "Delhi,45",
    "Ambala,35"
]

def mapper_avg_temp(lines):
    pairs = []
    for line in lines:
        parts = line.strip().rsplit(',', 1)
        if len(parts) == 2:
            city, temp = parts[0].strip(), float(parts[1].strip())
            pairs.append((city, temp))
    return pairs

def reducer_avg_temp(grouped):
    return {key: sum(values) / len(values) for key, values in grouped.items()}

mapped   = mapper_avg_temp(input_q8)
shuffled = shuffle(mapped)
result   = reducer_avg_temp(shuffled)

print("Q8 (A) Average Temperature Per City (Manual):")
for city, avg in sorted(result.items()):
    print(f"  {city}: {avg:.2f}°")

Q8 (A) Average Temperature Per City (Manual):
  Ambala: 35.00°
  Delhi: 45.00°
  London: 29.00°
  New York: 35.00°
  Tokyo: 35.00°


In [83]:
%%writefile q8_avg_temp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):

    def mapper(self, _, line):
        parts = line.strip().rsplit(',', 1)
        if len(parts) == 2:
            city = parts[0].strip()
            try:
                temp = float(parts[1].strip())
                yield city, temp
            except ValueError:
                pass

    def reducer(self, city, temps):
        temps = list(temps)
        yield city, sum(temps) / len(temps)

if __name__ == '__main__':
    MRAvgTemp.run()

Overwriting q8_avg_temp.py


In [84]:
with open('input_q8.txt', 'w') as f:
    f.write("New York,38\nLondon,29\nTokyo,35\nNew York,32\nDelhi,45\nAmbala,35\n")

print("Q8 (B) Average Temperature Per City (mrjob):")
!python q8_avg_temp.py input_q8.txt

Q8 (B) Average Temperature Per City (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8_avg_temp.root.20260425.095351.591150
Running step 1 of 1...
job output is in /tmp/q8_avg_temp.root.20260425.095351.591150/output
Streaming final output from /tmp/q8_avg_temp.root.20260425.095351.591150/output...
"London"	29.0
"New York"	35.0
"Tokyo"	35.0
"Ambala"	35.0
"Delhi"	45.0
Removing temp directory /tmp/q8_avg_temp.root.20260425.095351.591150...


---
## Q9. Average Temperature Per Country — Kaggle Global Land Temperatures Dataset

> Download the dataset from Kaggle and save `GlobalLandTemperaturesByCity.csv` in the same directory.

The relevant columns are `Country` and `AverageTemperature`. The mapper ignores rows with missing temperature values.

In [85]:
# ── (A) Manual MapReduce on Kaggle CSV ───────────────────────────────────────
import csv, os

KAGGLE_FILE = 'GlobalLandTemperaturesByCity.csv'

if not os.path.exists(KAGGLE_FILE):
    print("[!] Download the Kaggle dataset and place 'GlobalLandTemperaturesByCity.csv' here.")
else:
    # Read CSV lines
    kaggle_pairs = []
    with open(KAGGLE_FILE, 'r', encoding='utf-8', errors='ignore') as f:
        reader = csv.DictReader(f)
        for row in reader:
            country = row.get('Country', '').strip()
            temp_str = row.get('AverageTemperature', '').strip()
            if country and temp_str:
                try:
                    kaggle_pairs.append((country, float(temp_str)))
                except ValueError:
                    pass

    # Shuffle
    kaggle_grouped = defaultdict(list)
    for k, v in kaggle_pairs:
        kaggle_grouped[k].append(v)

    # Reduce
    kaggle_result = {k: sum(v) / len(v) for k, v in kaggle_grouped.items()}

    print("Q9 (A) Average Temperature Per Country (first 15):")
    for country, avg in sorted(kaggle_result.items())[:15]:
        print(f"  {country}: {avg:.2f}°C")

Q9 (A) Average Temperature Per Country (first 15):
  Afghanistan: 14.34°C
  Angola: 23.69°C
  Australia: 15.19°C
  Bangladesh: 25.49°C
  Brazil: 22.85°C
  Burma: 26.74°C
  Canada: 5.11°C
  Chile: 5.69°C
  China: 11.79°C
  Colombia: 20.89°C
  Congo (Democratic Republic Of The): 23.87°C
  Côte D'Ivoire: 26.16°C
  Dominican Republic: 25.98°C
  Egypt: 20.90°C
  Ethiopia: 17.53°C


In [86]:
%%writefile q9_avg_temp_country.py
from mrjob.job import MRJob

class MRAvgTempCountry(MRJob):

    def mapper(self, _, line):
        # Skip header
        if line.startswith('dt'):
            return
        parts = line.strip().split(',')
        # CSV columns: dt, AverageTemperature, AverageTemperatureUncertainty, City, Country, ...
        if len(parts) >= 5:
            temp_str = parts[1].strip()
            country  = parts[4].strip()
            if country and temp_str:
                try:
                    yield country, float(temp_str)
                except ValueError:
                    pass

    def reducer(self, country, temps):
        temps = list(temps)
        yield country, sum(temps) / len(temps)

if __name__ == '__main__':
    MRAvgTempCountry.run()

Overwriting q9_avg_temp_country.py


In [87]:
import os
if os.path.exists('GlobalLandTemperaturesByCity.csv'):
    print("Q9 (B) Average Temperature Per Country (mrjob):")
    !python q9_avg_temp_country.py GlobalLandTemperaturesByCity.csv
else:
    print("[!] Place 'GlobalLandTemperaturesByCity.csv' in this directory first.")

Q9 (B) Average Temperature Per Country (mrjob):
No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q9_avg_temp_country.root.20260425.095352.504905
Running step 1 of 1...
job output is in /tmp/q9_avg_temp_country.root.20260425.095352.504905/output
Streaming final output from /tmp/q9_avg_temp_country.root.20260425.095352.504905/output...
"Colombia"	20.892248063952035
"Congo (Democratic Republic Of The)"	23.866440922190204
"Dominican Republic"	25.976776619845943
"Egypt"	20.900406225165565
"Ethiopia"	17.52507266229899
"France"	10.402644346178143
"Germany"	8.916233733417561
"India"	25.80930923845554
"Indonesia"	26.65905694518361
"Iran"	12.57199211136891
"Iraq"	22.61434568965517
"Italy"	11.96550189513582
"Japan"	13.658246913580248
"Kenya"	16.081395113230034
"Mexico"	15.717422377622377
"Morocco"	17.184157858613588
"Nigeria"	26.36132269230769
"Pakistan"	24.811197226502312
"Peru"	16.769119658119656
"Philippines"	26.4483344878